In [1290]:
import pandas as pd
from functools import lru_cache
import ast
import re

import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


# Loading data into separate df's 

In [1291]:
df_1 = pd.read_csv('/Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/csv/gpt_baseline_1.csv')
# df_1.isnull().sum()   
print("original number of rows: ", len(df_1))

original number of rows:  66


#### Preprocessing

In [1292]:
#Removing spaces from the columns
df_1['KG_step'] = df_1['KG_step'].str.replace(' ', '')
df_1['student_step'] = df_1['student_step'].str.replace(' ', '')
df_1['student_update_plan'] = df_1['student_update_plan'].str.replace(' ', '')
# remove '(' and ')' from the student_step and student_update_plan and KG_step
df_1['student_step'] = df_1['student_step'].str.replace('(', '').str.replace(')', '')
df_1['student_update_plan'] = df_1['student_update_plan'].str.replace('(', '').str.replace(')', '')
df_1['KG_step'] = df_1['KG_step'].str.replace('(', '').str.replace(')', '')


In [1293]:
# count rows where student_distance == 0
print("number of rows where student_distance == 0: ", len(df_1[df_1['student_distance'] == 0]))
# show the rows where student_distance == 0
print(df_1[df_1['student_distance'] == 0][['conclusion', 'student_step', 'student_distance']])
# add 7 to the student_distance column
df_1['student_distance'] = df_1['student_distance'].fillna(0)
# replace 0 with 7 in the student_distance column
df_1['student_distance'] = df_1['student_distance'].replace(0, 7)
print(df_1[df_1['student_distance'] == 0][['conclusion', 'student_step', 'student_distance']])


number of rows where student_distance == 0:  1
   conclusion student_step  student_distance
56      (Y>C)          Y>P                 0
Empty DataFrame
Columns: [conclusion, student_step, student_distance]
Index: []


# Optimal comparison 

In [1294]:
# remove ( and ) from student_step and KG_step
df_1['student_step'] = df_1['student_step'].str.replace('(', '').str.replace(')', '')
df_1['KG_step'] = df_1['KG_step'].str.replace('(', '').str.replace(')', '')
#count the number of rows where student_step != KG_step
df_1['count'] = df_1.apply(lambda row: 1 if (row['student_step'] == row['KG_step']) & (row['student_rule'] == row['KG_rule']) else 0, axis=1)
print("Optimal number of steps: ", df_1['count'].sum())
df_1_opt = df_1[df_1['student_step'] == df_1['KG_step']].copy()

Optimal number of steps:  6


In [1295]:

# mean complexity of the optimal steps
print("mean of the complexity of the optimal steps: ", df_1_opt['student_complexity'].mean())
print("mean of the complexity of the actual steps: ", df_1_opt['KG_complexity'].mean())
# mean of the already derived statements
print("total statements: ", (df_1_opt['length_givens'] + df_1_opt['length_intermediate_expressions']).mean())
print("already derived statements: ", (df_1_opt['length_intermediate_expressions']).mean())
# replace the None values in the student_depth column with 1
df_1_opt['student_depth'] = df_1_opt['student_depth'].fillna(1)
# mean of the student depth
print("student depth: ", df_1_opt['student_depth'].mean())
print("KG depth: ", df_1_opt['KG_depth'].mean())

print("optimal steps distance: ", df_1_opt['student_distance'].mean())
print("optimal steps distance kg: ", df_1_opt['KG_distance'].mean())

# sum of mean of student_parent_complexities
df_1_opt['student_parent_complexities'] = df_1_opt['student_parent_complexities'].apply(ast.literal_eval)
sum_of_means = [sum(x)/len(x) if len(x) > 0 else 0 for x in df_1_opt['student_parent_complexities']]
print("sum of mean of student_parent_complexities: ", sum(sum_of_means)/len(df_1_opt))

# df_1_opt[['student_step', 'KG_step','KG_distance', 'student_distance']]
# Note: if student_step == KG_step, then student_distance == KG_distance


mean of the complexity of the optimal steps:  1.9
mean of the complexity of the actual steps:  1.9
total statements:  7.1
already derived statements:  4.1
student depth:  1.0
KG depth:  1.0
optimal steps distance:  -0.2
optimal steps distance kg:  1.3
sum of mean of student_parent_complexities:  2.3


# Suboptimal Comprison 

In [1296]:
# Remove optimal step rows 
df_1_not_opt = df_1[df_1['student_step'] != df_1['KG_step']].copy()
print("number of rows after removing matching steps: ", len(df_1_not_opt))

number of rows after removing matching steps:  56


In [1297]:

# construct suboptimal df
df_1_not_opt['filtered_derivation_keys'] = df_1_not_opt['filtered_derivation_keys'].apply(ast.literal_eval) # Parse string lists to actual lists (do this once)
df_1_not_opt['count'] = [int(step in keys) for step, keys in zip(df_1_not_opt['student_step'], df_1_not_opt['filtered_derivation_keys'])] # Use list comprehension (faster than apply)
print("suboptimal steps: ", df_1_not_opt['count'].sum())
df_1_sub = df_1_not_opt[df_1_not_opt['count'] == 1].copy() # only those rows where student_step is in filtered_derivation_keys
# Fix: assign the filled values back (the previous line didn't modify the dataframe)
df_1_sub['student_depth'] = df_1_sub['student_depth'].fillna(7)
# df_1_sub[['student_step', 'KG_step', 'student_depth', 'student_update_plan']]

# mean complexity of the optimal steps
print("mean of the complexity of the suboptimal steps: ", df_1_sub['student_complexity'].mean())
print("mean of the complexity of the actual steps: ", df_1_sub['KG_complexity'].mean())
# mean of the already derived statements
print("total statements: ", (df_1_sub['length_givens'] + df_1_sub['length_intermediate_expressions']).mean())
print("already derived statements: ", (df_1_sub['length_intermediate_expressions']).mean())
# replace the None values in the student_depth column with 1
# df_1_opt['student_depth'] = df_1_opt['student_depth'].fillna(1)
# mean of the student depth
print("student depth: ", df_1_sub['student_depth'].mean())
print("KG depth: ", df_1_sub['KG_depth'].mean())

print("optimal steps distance: ", df_1_sub['student_distance'].mean())
print("optimal steps distance kg: ", df_1_sub['KG_distance'].mean())

# sum of mean of student_parent_complexities
df_1_sub['student_parent_complexities'] = df_1_sub['student_parent_complexities'].apply(ast.literal_eval)
sum_of_means = [sum(x)/len(x) if len(x) > 0 else 0 for x in df_1_sub['student_parent_complexities']]
print("sum of mean of student_parent_complexities: ", sum(sum_of_means)/len(df_1_opt))


suboptimal steps:  4
mean of the complexity of the suboptimal steps:  0.0
mean of the complexity of the actual steps:  5.25
total statements:  7.0
already derived statements:  4.0
student depth:  4.75
KG depth:  1.0
optimal steps distance:  -3.0
optimal steps distance kg:  4.25
sum of mean of student_parent_complexities:  1.35


### Suboptimal at depth ==1 

In [1298]:
# suboptimal df (where student_step is in filtered_derivation_keys and depth is 1)
df_1_sub_1 = df_1_sub[df_1_sub['student_depth'] == 1].copy()
print("suboptimal steps with depth 1: ", len(df_1_sub_1))
# df_1_sub_1[['student_step', 'KG_step', 'student_depth', 'student_update_plan']]



suboptimal steps with depth 1:  0


### Suboptimal at depth > 1 

In [1299]:
df_1_sub_not_1 = df_1_sub[df_1_sub['student_depth'] > 1].copy()
print("suboptimal steps with depth > 1: ", len(df_1_sub_not_1))
# df_1_sub_not_1[['student_step', 'KG_step', 'student_depth', 'student_update_plan']]
# count the updates here 

suboptimal steps with depth > 1:  4


# Incorrect steps

In [1300]:
# incorrect steps
df_1_in = df_1_not_opt[df_1_not_opt['count'] == 0].copy()
print("incorrect steps: ", len(df_1_in))
# df_1_in[['student_step', 'KG_step', 'student_update_plan']]

df_1_in['student_depth'] = df_1_in['student_depth'].fillna(7)
# df_1_sub[['student_step', 'KG_step', 'student_depth', 'student_update_plan']]

# mean complexity of the optimal steps
print("mean of the complexity of the incorrect steps: ", df_1_in['student_complexity'].mean())
print("mean of the complexity of the actual steps: ", df_1_in['KG_complexity'].mean())
# mean of the already derived statements
print("total statements: ", (df_1_in['length_givens'] + df_1_in['length_intermediate_expressions']).mean())
print("already derived statements: ", (df_1_sub['length_intermediate_expressions']).mean())
# replace the None values in the student_depth column with 1
# df_1_opt['student_depth'] = df_1_opt['student_depth'].fillna(1)
# mean of the student depth
print("student depth: ", df_1_in['student_depth'].mean())
print("KG depth: ", df_1_in['KG_depth'].mean())

print("incorrect steps distance: ", df_1_in['student_distance'].mean())
print("incorrect steps distance kg: ", df_1_in['KG_distance'].mean())

# sum of mean of student_parent_complexities
df_1_in['student_parent_complexities'] = df_1_in['student_parent_complexities'].apply(ast.literal_eval)
sum_of_means = [sum(x)/len(x) if len(x) > 0 else 0 for x in df_1_in['student_parent_complexities']]
print("sum of mean of student_parent_complexities: ", sum(sum_of_means)/len(df_1_opt))


incorrect steps:  52
mean of the complexity of the incorrect steps:  2.7884615384615383
mean of the complexity of the actual steps:  4.288461538461538
total statements:  6.826923076923077
already derived statements:  4.0
student depth:  0.11538461538461539
KG depth:  1.0
incorrect steps distance:  1.3461538461538463
incorrect steps distance kg:  2.2884615384615383
sum of mean of student_parent_complexities:  19.85


# Correction after update

In [1301]:
#remove ( and ) from student_update_plan and KG_step
df_1_not_opt['student_update_plan'] = df_1_not_opt['student_update_plan'].str.replace('(', '').str.replace(')', '')
df_1_not_opt['KG_step'] = df_1_not_opt['KG_step'].str.replace('(', '').str.replace(')', '')
df_1_not_opt['count'] = df_1_not_opt.apply(lambda row: 1 if (row['student_update_plan'] == row['KG_step']) else 0, axis=1)
print("student update plan that match KG steps: ", df_1_not_opt['count'].sum())
df_1_update = df_1_not_opt[df_1_not_opt['count'] == 1].copy()
df_1_update[['student_step', 'student_rule', 'KG_rule', 'student_update_rule', 'student_update_plan', 'KG_step', ]]


student update plan that match KG steps:  13


,student_step,student_rule,KG_rule,student_update_rule,student_update_plan,KG_step
0,W,DS,Add,Add,H+F,H+F
4,W,DS,Distributive,Distribution,W+-H*W+-F,W+-H*W+-F
9,-H*-F,DN,MT,DS,--W,--W
12,N>-J,HS,Simp,Equivalence,C>-J,C>-J
20,N>-J,HS,Simp,Equivalence,C>-J,C>-J
21,N>-J,HS,Simp,Equivalence,C>-J,C>-J
24,Y>C,HS,Simp,Equivalence,Y>P,Y>P
27,P>C,Impl,Simp,Contraposition,-C>-P,-C>-P
31,Y>-C,Equiv,Simp,Simplification,-P>-C,-P>-C
45,Y>C,HS,Simp,Equivalence,Y>P,Y>P


### Incorrect rows after update

In [1302]:
# Simple way: rows where student_step != KG_step but student_update_plan == KG_step
df_1_not_update = df_1_not_opt[df_1_not_opt['student_update_plan'] != df_1_not_opt['KG_step']].copy()
print("No updates: ", len(df_1_not_update))
df_1_not_update[['student_step', 'KG_step', 'student_update_plan']]


df_1_not_update['student_depth'] = df_1_not_update['student_depth'].fillna(7)
# df_1_sub[['student_step', 'KG_step', 'student_depth', 'student_update_plan']]

# mean complexity of the optimal steps
print("mean of the complexity of the suboptimal steps: ", df_1_not_update['student_complexity'].mean())
print("mean of the complexity of the actual steps: ", df_1_not_update['KG_complexity'].mean())
# mean of the already derived statements
print("total statements: ", (df_1_not_update['length_givens'] + df_1_not_update['length_intermediate_expressions']).mean())
print("already derived statements: ", (df_1_not_update['length_intermediate_expressions']).mean())
# replace the None values in the student_depth column with 1
# df_1_opt['student_depth'] = df_1_opt['student_depth'].fillna(1)
# mean of the student depth
print("student depth: ", df_1_not_update['student_depth'].mean())
print("KG depth: ", df_1_not_update['KG_depth'].mean())

print("optimal steps distance: ", df_1_not_update['student_distance'].mean())
print("optimal steps distance kg: ", df_1_not_update['KG_distance'].mean())

# sum of mean of student_parent_complexities
df_1_not_update['student_parent_complexities'] = df_1_not_update['student_parent_complexities'].apply(ast.literal_eval)
sum_of_means = [sum(x)/len(x) if len(x) > 0 else 0 for x in df_1_not_update['student_parent_complexities']]
print("sum of mean of student_parent_complexities: ", sum(sum_of_means)/len(df_1_opt))
df_1_not_update[['student_step', 'KG_step', 'student_update_plan', 'teacher_feedback']]



No updates:  43
mean of the complexity of the suboptimal steps:  2.697674418604651
mean of the complexity of the actual steps:  4.674418604651163
total statements:  6.744186046511628
already derived statements:  3.744186046511628
student depth:  0.13953488372093023
KG depth:  1.0
optimal steps distance:  1.813953488372093
optimal steps distance kg:  2.604651162790698
sum of mean of student_parent_complexities:  17.2


,student_step,KG_step,student_update_plan,teacher_feedback
2,W,--H+F,-H+F,"Great job identifying that (W+-(H+F)) and -(H+F) would allow you to use DS to reach the conclusion. However, you need to first derive -(H+F) before applying DS. Which of your candidates helps you establish this missing piece?"
3,--H*-F,W+-H+F,W+-H*-F,"Great job identifying that Disjunctive Syllogism could help reach the conclusion. However, check if your candidate can be directly derived from your available statements using the rules and whether DN is the right rule to use here. Is there a way to combine your known statements to form a disjunction that matches the structure needed for DS?"
5,-H*-F,--H+F,-H+F,"Great job identifying that deriving (-H*-F) would help you reach W. However, check if your chosen rule and parent statement actually allow you to derive a conjunction from a disjunction. Is there a transformation that lets you rewrite (H+F) in a form closer to (-H*-F)?"
6,W,W+-H,W+-F,"Great job identifying the need to use DS, but notice that (W+(-H*-F)) is not a simple disjunction with H as a direct negation. Can you simplify (W+(-H*-F)) using implication rules before applying DS? Which candidate might help you break down the structure further?"
7,--H*-F,W+-H+F,NoImprovementNeeded,Great job recognizing the need to use De Morgan's Law here! How will this new statement help you apply Disjunctive Syllogism to reach the conclusion W?
13,-N>-J,-J>C*C>-J,-J>C,"Great job identifying the use of HS and aiming for a conditional form! However, before applying HS, consider if you have all the necessary components explicitly available. Is there a way to combine or extract information from the biconditional to set up for HS more directly? Look at your candidate (-J>C) and think about how it might help."
14,C>-J,C>-J*-J>C,NoImprovementNeeded,"Great job identifying the need to convert the biconditional into implications! As you proceed, consider how you might use this new implication with your other derived statements to get closer to the conclusion. What rule could help you connect these implications?"
16,C>-J,C>-J*-J>C,NoImprovementNeeded,"Great job! Converting the biconditional to an implication is a strong move here. As you proceed, consider how this new implication might connect with your other givens to get closer to the conclusion. What could you do next to relate N to -J?"
22,-Y=-P,-P>-C*-C>-P,C=P,"Great job recognizing the usefulness of contraposition! However, consider which equivalence, when contraposed, will help you connect more directly to the other statements and move toward the conclusion. Which statement, if rewritten, would better link the negative terms you see in the givens?"
25,Y>P,-C>-P,-P>-C,"Great job recognizing the importance of implication direction for reaching the conclusion. Consider which available equivalence more directly connects to C, and whether isolating a statement involving C might better set up your next step. Which candidate would help you build a chain toward (Y>C) using the givens?"


# ------

In [1303]:
# Loading data into separate df's 
df_1 = pd.read_csv('/Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/csv/gpt_ours.csv')
# df_1.isnull().sum()   
print("original number of rows: ", len(df_1))
#### Preprocessing
#Removing spaces from the columns
df_1['KG_step'] = df_1['KG_step'].str.replace(' ', '')
df_1['student_step'] = df_1['student_step'].str.replace(' ', '')
df_1['student_update_plan'] = df_1['student_update_plan'].str.replace(' ', '')
# remove starting '(' and ending ')' from the student_step and student_update_plan and KG_step
# df_1['student_step'] = df_1['student_step'].str.replace('(', '').str.replace(')', '')
# df_1['student_update_plan'] = df_1['student_update_plan'].str.replace('(', '').str.replace(')', '')
# df_1['KG_step'] = df_1['KG_step'].str.replace('(', '').str.replace(')', '')

# count rows where student_distance == 0
print("number of rows where student_distance == 0: ", len(df_1[df_1['student_distance'] == 0]))
# show the rows where student_distance == 0
print(df_1[df_1['student_distance'] == 0][['conclusion', 'student_step', 'student_distance']])


# Optimal comparison 
df_1.columns
# remove ( and ) from student_step and KG_step
df_1['student_step'] = df_1['student_step'].str.replace('(', '').str.replace(')', '')
df_1['KG_step'] = df_1['KG_step'].str.replace('(', '').str.replace(')', '')
#count the number of rows where student_step != KG_step
df_1['count'] = df_1.apply(lambda row: 1 if row['student_step'] == row['KG_step'] else 0, axis=1)
print("Optimal number of steps: ", df_1['count'].sum())
df_1_opt = df_1[df_1['student_step'] == df_1['KG_step']].copy()
# mean complexity of the optimal steps
print("mean of the complexity of the optimal steps: ", df_1_opt['student_complexity'].mean())
print("mean of the complexity of the actual steps: ", df_1_opt['KG_complexity'].mean())
# mean of the already derived statements
print("already derived statements: ", (df_1_opt['length_givens'] + df_1_opt['length_intermediate_expressions']).mean())
# replace the None values in the student_depth column with 1
df_1_opt['student_depth'] = df_1_opt['student_depth'].fillna(1)
# mean of the student depth
print("student depth: ", df_1_opt['student_depth'].mean())
print("KG depth: ", df_1_opt['KG_depth'].mean())

# sum of mean of student_parent_complexities
df_1_opt['student_parent_complexities'] = df_1_opt['student_parent_complexities'].apply(ast.literal_eval)
sum_of_means = [sum(x)/len(x) if len(x) > 0 else 0 for x in df_1_opt['student_parent_complexities']]
print("sum of mean of student_parent_complexities: ", sum(sum_of_means)/len(df_1_opt))

# df_1_opt[['student_step', 'KG_step','KG_depth', 'student_depth']]
# Suboptimal Comprison 
# Remove optimal step rows 
df_1_not_opt = df_1[df_1['student_step'] != df_1['KG_step']].copy()
print("number of rows after removing matching steps: ", len(df_1_not_opt))
# construct suboptimal df
df_1_not_opt['filtered_derivation_keys'] = df_1_not_opt['filtered_derivation_keys'].apply(ast.literal_eval) # Parse string lists to actual lists (do this once)
df_1_not_opt['count'] = [int(step in keys) for step, keys in zip(df_1_not_opt['student_step'], df_1_not_opt['filtered_derivation_keys'])] # Use list comprehension (faster than apply)
print("suboptimal steps: ", df_1_not_opt['count'].sum())
df_1_sub = df_1_not_opt[df_1_not_opt['count'] == 1].copy() # only those rows where student_step is in filtered_derivation_keys
# Fix: assign the filled values back (the previous line didn't modify the dataframe)
df_1_sub['student_depth'] = df_1_sub['student_depth'].fillna(7)
# df_1_sub[['student_step', 'KG_step', 'student_depth', 'student_update_plan']]

### Suboptimal at depth ==1 
# suboptimal df (where student_step is in filtered_derivation_keys and depth is 1)
df_1_sub_1 = df_1_sub[df_1_sub['student_depth'] == 1].copy()
print("suboptimal steps with depth 1: ", len(df_1_sub_1))
# df_1_sub_1[['student_step', 'KG_step', 'student_depth', 'student_update_plan']]


### Suboptimal at depth > 1 
df_1_sub_not_1 = df_1_sub[df_1_sub['student_depth'] > 1].copy()
print("suboptimal steps with depth > 1: ", len(df_1_sub_not_1))
# df_1_sub_not_1[['student_step', 'KG_step', 'student_depth', 'student_update_plan']]
# count the updates here 
# Incorrect steps
# incorrect steps
df_1_in = df_1_not_opt[df_1_not_opt['count'] == 0].copy()
print("incorrect steps: ", len(df_1_in))
# df_1_in[['student_step', 'KG_step', 'student_update_plan']]


# Correction after update
#remove ( and ) from student_update_plan and KG_step
df_1_not_opt['student_update_plan'] = df_1_not_opt['student_update_plan'].str.replace('(', '').str.replace(')', '')
df_1_not_opt['KG_step'] = df_1_not_opt['KG_step'].str.replace('(', '').str.replace(')', '')
df_1_not_opt['count'] = df_1_not_opt.apply(lambda row: 1 if row['student_update_plan'] == row['KG_step'] else 0, axis=1)
print("student update plan that match KG steps: ", df_1_not_opt['count'].sum())
df_1_update = df_1_not_opt[df_1_not_opt['count'] == 1].copy()
df_1_update[['student_step', 'KG_step', 'student_update_plan', 'judge_feedback']].loc[0:5]

### Incorrect rows after update
# Simple way: rows where student_step != KG_step but student_update_plan == KG_step
df_1_not_update = df_1_not_opt[df_1_not_opt['student_update_plan'] != df_1_not_opt['KG_step']].copy()
print("No updates: ", len(df_1_not_update))
df_1_not_update[['student_step', 'KG_step', 'student_update_plan', 'final_feedback']].loc[0:5]
df_1_update[['student_step', 'KG_step', 'student_update_plan', 'final_feedback']].loc[0:5]


original number of rows:  66
number of rows where student_distance == 0:  1
   conclusion student_step  student_distance
56      (Y>C)        (Y>P)                 0
Optimal number of steps:  10
mean of the complexity of the optimal steps:  1.9
mean of the complexity of the actual steps:  1.9
already derived statements:  7.1
student depth:  1.0
KG depth:  1.0
sum of mean of student_parent_complexities:  2.3
number of rows after removing matching steps:  56
suboptimal steps:  4
suboptimal steps with depth 1:  0
suboptimal steps with depth > 1:  4
incorrect steps:  52
student update plan that match KG steps:  36
No updates:  20


KeyError: "['final_feedback'] not in index"